In [34]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel,Field
import operator

In [20]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task= "text-generation",
)

model = ChatHuggingFace(llm=llm)

In [21]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)
    

In [28]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(
    pydantic_object=EvaluationSchema
)

In [29]:
essay = """
Artificial Intelligence (AI) is becoming an important part of the modern world. It is being used in healthcare, education, agriculture, banking, transportation, defence, and many other fields. India has the potential to become one of the leading countries in AI because of its large population, strong IT industry, skilled workforce, and growing technology ecosystem.

India is using AI to solve real-world problems. In healthcare, AI can help doctors analyze medical images, identify diseases, and improve patient care. In agriculture, AI can help farmers predict weather, identify crop diseases, and improve the use of water and fertilizers. In education, AI can provide personalized learning and help students learn according to their individual needs.

India also has a large pool of software engineers, researchers, and technology professionals. Indian IT companies and startups are developing AI-based products and services for both Indian and international markets. Universities and research institutions are also working on AI research and training students in machine learning and related technologies.

Another important role of India is developing AI for Indian languages and local communities. India has many languages and a large population that may not always use English for digital services. AI can help create voice assistants, translation systems, and other digital tools that work in Indian languages.

The Indian government is also supporting AI development through initiatives such as the IndiaAI Mission. These efforts aim to improve AI infrastructure, research, skills, innovation, and responsible use of AI.

However, India also faces challenges. These include the shortage of advanced AI computing resources, the need for more AI researchers, data privacy concerns, cybersecurity risks, and the possibility of job displacement due to automation. India needs proper regulations, education, and responsible AI development to address these challenges.

In conclusion, India can play a major role in the global AI ecosystem. With its large talent pool, growing startup ecosystem, strong IT industry, and government support, India has the opportunity to develop AI solutions that benefit both India and the world. The future of AI in India will depend on innovation, skilled professionals, responsible development, and effective use of technology.
"""

In [30]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
Evaluate the language quality of the following essay.

Give:
1. Detailed feedback
2. Score out of 10

{format_instructions}

Essay:
{essay}
""",
    input_variables=["essay"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [31]:
chain = prompt | model | parser

In [32]:
result = chain.invoke({
    "essay": essay
})

print(result)

{'feedback': 'The essay is well-structured and covers various aspects of AI in India, providing specific examples and discussing both opportunities and challenges. However, the language could be more varied and engaging, and some sentences could be more concise. The essay lacks minor grammatical errors and maintains clarity, but there could be a stronger emphasis on connecting ideas for smoother transitions. The tone is generally academic but could benefit from a slightly more dynamic approach in certain sections.', 'score': 8}


In [35]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [38]:
def evaluate_language(state: UPSCState):

    prompt = f"""
    Evaluate the language quality of the following essay.

    Provide detailed feedback and assign a score out of 10.

    Essay:
    {state["essay"]}
    """

    output = chain.invoke({
        "essay": state["essay"]
    })

    return {
        "language_feedback": output["feedback"],
        "individual_scores": [output["score"]]
    }

In [ ]:
def evaluate_analysis(state: UPSCState)-> UPSCState:

In [ ]:
def evaluate_thought(state: UPSCState)-> UPSCState:

In [ ]:
def final_evaluation(state: UPSCState)-> UPSCState:

In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluate)

NameError: name 'evaluate_language' is not defined